# 🔐 **Security Evaluation of a Face Recognition System**

## Detector Training Defense Strategy Implementation for NN1

**Academic Year:** 2024-2025  
**Group:** 04

---

### 👥 Team Members
- **Agostino Cardamone** — `0622702276`
- **Asja Antonucci**     — `0622702437`
- **Chiara Ferraioli**   — `0622702169`

---

### 📚 Overview of This Section

1. [Setup and Data Loading](#1-setup-and-data-loading)
2. [Detector of Adversarial Samples Defense Strategy](#2-detector-adversarial-samples-defense-strategy)

## 1. Setup and Data Loading

#### Environment Setup

To ensure reproducibility and avoid package conflicts, it is strongly recommended to run all experiments in an isolated environment. We use Conda to create and manage the project environment, and all Python dependencies are listed in the requirements.txt file.

In [ ]:
# 1) Create a new environment named “aic_env”
#conda create -n aic_env python=3.10 -y

# 2) Switch into the new environment
# On Windows:
# conda activate aic_env
# On Linux/macOS:
# source activate aic_env

# 3) Install all dependencies
#!pip install -r requirements.txt

import os 

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch
print("torch.version:", torch.__version__)
print("CUDA available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

#### Dataset Configuration and Paths

This final notebook begins with the set up of the environment. Paths and directories are established for managing datasets, including the `VGGFace2 dataset` and folders for clean and adversarial samples. The configuration also determines whether to build a new test set, apply `MTCNN preprocessing`, or load existing CSV files without modifying images. Additional folders are created to store generated adversarial examples.

In [ ]:
from utils import *         # Project-specific utilities and imports

# —————————————————————————————————————————————
#           Paths and configuration
# —————————————————————————————————————————————

# Base directory containing all dataset-related files
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# If True, a new test set will be built by sampling and copying images from the original VGGFace2 dataset
# NOTE: This requires the dataset to be downloaded and extracted under 'vggface2_train/trainset'
# If False, the existing CSV files will be loaded without modifying or copying any images
dataset_selection = False

# If True, test images will be aligned and cropped using MTCNN preprocessing (for NN1 compatibility)
mtcnn_processing = False

# Path to VGGFace2 identity metadata (includes Class_ID, Name, Gender, etc.)
vgg2_dataset_annotations_path = os.path.join(dataset_dir, 'identity_meta.csv')

# Folder structure for test set files and images
test_set_folder              = os.path.join(dataset_dir, 'testset')
test_set_data_folder         = os.path.join(test_set_folder, 'samples')
test_set_annotations_folder  = os.path.join(test_set_folder, 'test_set.csv')


# Training set folders and output path
train_set_folder = os.path.join(dataset_dir, 'trainset')
os.makedirs(train_set_folder, exist_ok=True)

train_set_clean_folder = os.path.join(train_set_folder, "clean_samples")
os.makedirs(train_set_clean_folder, exist_ok=True)

train_set_adversarial_folder = os.path.join(train_set_folder, "adversarial_samples")
os.makedirs(train_set_adversarial_folder, exist_ok=True)

# Directory to save generated adversarial examples
adversarial_folder = os.path.join(os.getcwd(), 'attacks')

# Device configuration: use GPU if available, otherwise fallback to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

After discovering that the pre-processing pipeline alone was mainly effective against non-gradient-based attacks like Carlini-Wagner, we proceeded to implement an alternative, **optional defence strategy** outlined in the project specifications: training a dedicated adversarial detector. This detector aims to **explicitly identify adversarial examples**, thereby extending the model's robustness beyond the pre-processing approach.

To set this up, we leveraged and modified parts of the existing data handling code already used for the base NN1 evaluation. Specifically, we adapted the sampling procedures to build a balanced training set for the binary input detector. 

The dataset for the detector was built by:
- **Extracting clean samples** directly from the original VGGFace2 training images, ensuring no overlap with the test identities.
- **Collecting adversarial examples** generated during prior attack experiments.
- Ensuring a balanced mix of **clean vs adversarial samples**, each labelled accordingly for binary classification.

The implementation draws on the same structured data processing and CSV-based class information already present in the pipeline. In this new context, it enables the creation of a **detector dataset** consistent with the overall project workflow.

In [ ]:
import random

# ————————————————————————————————————————————
#  Download and load the class‐label mapping
# ————————————————————————————————————————————

# Our face‐recognition model expects a NumPy array of all 8631 labels.
# We download it from the official rcmalli/keras-vggface repo if missing.
labels_url = (
    "https://github.com/rcmalli/keras-vggface/"
    "releases/download/v2.0/rcmalli_vggface_labels_v2.npy"
)
labels_path = os.path.join(dataset_dir, 'rcmalli_vggface_labels_v2.npy')

# Ensure the directory for labels_path exists
os.makedirs(os.path.dirname(labels_path), exist_ok=True)

# Download only if the file does not already exist on disk
if not os.path.exists(labels_path):
    print(f"Downloading LABELS to {labels_path}…")
    urllib.request.urlretrieve(labels_url, labels_path)

# Load the .npy file into a NumPy array and strip any padding whitespace
LABELS = np.load(labels_path)
LABELS = np.char.strip(LABELS)

# ———————————————————————————————————————————————————————————————————————
#   Generate training set (clean samples only) for BinaryInputDetector
# ———————————————————————————————————————————————————————————————————————

# Source root for the full VGGFace2 train images on your machine
train_root = (
    #"C:/Users/chiar/Desktop/Uni/AI for Cybersecurity/AI-for-Cybersecurity-Project/dataset/vggface2_train/train"  # e.g., "/home/user/datasets/vggface2_train/trainset"
    "C:/Users/agost/Documents/GitHub/AI-for-Cybersecurity-Project/dataset/vggface2_train/train"
)

# Read the full VGGFace2 metadata CSV into a DataFrame (contains 9131 classes)
vgg2_dataset = pd.read_csv(
    filepath_or_buffer=vgg2_dataset_annotations_path,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

test_set = pd.read_csv(
    filepath_or_buffer=test_set_annotations_folder,
    sep=',',
    skipinitialspace=True,
    engine='python'
)

test_set_size = len(test_set) * 10

test_ids = test_set['Class_ID'].tolist()
    
if dataset_selection:
    
    def try_add_samples(df, max_count, dest_folder):
        
        added = 0
        
        for _, row in df.iterrows():
            if added >= max_count:
                break
            
            class_id = row['Class_ID']
            
            src_dir = os.path.join(train_root, class_id)

            if not os.path.exists(src_dir):
                continue
            all_images = os.listdir(src_dir)
            if len(all_images) < 1:
                continue
            if class_id in test_ids:
                continue
            
            try:
                added += 1
                img = random.choice(all_images)
                src_path = os.path.join(src_dir, img)
                dst_path = os.path.join(dest_folder, f'{added:04d}.jpg')
                shutil.copy(src_path, dst_path)
            except Exception as e:
                print(f"Error: could not copy for {class_id} → {e}")
                added -= 1
    
    try_add_samples(vgg2_dataset.sample(frac=1, random_state=42), max_count=2000, dest_folder=train_set_clean_folder)
    
    try_add_samples(vgg2_dataset.sample(frac=1, random_state=43), max_count=2000, dest_folder=train_set_adversarial_folder)

Continuing with the implementation of the detector strategy, the **MTCNN-based face alignment and cropping** remains an important step, but is adapted for the new dataset structure dedicated to training the detector. 

Previously, the MTCNN was used to **align test images** directly within the evaluation pipeline of `NN1`, ensuring that the test set images were properly standardised before classification. Here, the **same MTCNN-based pre-processing** is reused but applied to the **newly curated clean and adversarial training images** separately.

In [ ]:
# ———————————————————————————————————————————————————
#  Initialize the face detector and aligner (MTCNN)
# ———————————————————————————————————————————————————
# We use MTCNN from facenet-pytorch to detect, crop, and align faces in one step.
# When you call face_detector_nn1(img_batch), it returns a tensor of shape [B, 3, image_size, image_size]
# containing the aligned face crops, ready to feed into nn1 or an adversarial attack.

face_detector_nn1 = MTCNN(
    image_size=160,                             # int: output height/width of each face crop (default=160)
    margin=0,                                   # int: number of pixels to expand the face bounding box (default=0)
    min_face_size=20,                           # int: minimum face size (in pixels) that the detector will attempt to locate (default=20)
    thresholds=[0.6, 0.7, 0.7],                 # list of 3 floats: score thresholds for each detection stage—
                                                #   P-Net, R-Net, and O-Net respectively (default=[0.6, 0.7, 0.7])
    factor=0.709,                               # float: scale factor between pyramid levels; controls the search granularity (default=0.709)
    post_process=True,                          # bool: whether to apply face alignment post-processing (True)
    select_largest=True,                        # bool: if multiple faces are detected, return only the largest one (True)
    selection_method="center_weighted_size",    # str: heuristic for choosing among multiple detections—
                                                #   options include "largest" or "center_weighted_size" (default="center_weighted_size")
    keep_all=False,                             # bool: if True, return all detected faces; if False, return only one (default=False)
    device=device                               # torch.device or str: computation device, e.g. "cuda:0" or "cpu"
)

The logic of detection and alignment remains unchanged, ensuring consistent pre-processing across both the original classification model and the adversarial detector.

Specifically, the MTCNN is called on each image in the clean and adversarial sets. Cropped and aligned images are saved to designated folders for clean and adversarial examples, respectively (`cropped_faces_clean` and `cropped_faces_adv`). Counts of successfully processed images are tracked, and any imbalance in the number of aligned clean vs adversarial samples is adjusted by removing the excess from the larger set to ensure that the final detector dataset is balanced.

Finally, the **aligned face crops** are **stacked into tensors** and saved as separate datasets for future use (`clean_train_set.pt`, `adv_train_set.pt`). A single `DataLoader` for the detector is also built and saved (`dataloader_detector.pt`), ready for training the adversarial detector model.

This approach maintains consistency in image alignment across all stages of the project while tailoring the pre-processing specifically for the binary detector task.

In [ ]:
if mtcnn_processing:
    no_clean_face = 0
    clean_train_set = []
    
    cropped_clean = os.path.join(train_set_folder, 'cropped_faces_clean')
    os.makedirs(cropped_clean, exist_ok=True)
    
    if os.path.isdir(cropped_clean) and os.listdir(cropped_clean):
        logger.info(f"Found existing crops in {cropped_clean}, skipping face cropping.")
    else:
        all_images = os.listdir(train_set_clean_folder)
        
        for i, img in enumerate(all_images):
            
            src_path = os.path.join(train_set_clean_folder, img)
            
            save_path = os.path.join(cropped_clean, img)
            
            img_aligned = face_detector_nn1(Image.open(src_path), return_prob = False, save_path=save_path)
            
            if img_aligned is not None:
                clean_train_set.append(img_aligned)
            else:
                logger.info(f'Face not detected in image {i}')
                no_clean_face += 1
    
    no_adv_face = 0
    adv_train_set = []
    
    cropped_adv = os.path.join(train_set_folder, 'cropped_faces_adv')
    os.makedirs(cropped_adv, exist_ok=True)
    
    if os.path.isdir(cropped_adv) and os.listdir(cropped_adv):
        logger.info(f"Found existing crops in {cropped_adv}, skipping face cropping.")
    else:
        all_images = os.listdir(train_set_adversarial_folder)
        
        for i, img in enumerate(all_images):
            
            src_path = os.path.join(train_set_adversarial_folder, img)
            
            save_path = os.path.join(cropped_adv, img)
            
            img_aligned = face_detector_nn1(Image.open(src_path), return_prob = False, save_path=save_path)
            
            if img_aligned is not None:
                adv_train_set.append(img_aligned)
            else:
                logger.info(f'Face not detected in image {i}')
                no_adv_face += 1
    if (len(clean_train_set) != 0 and len(adv_train_set) != 0):
        
        if(no_clean_face > no_adv_face):
            to_remove = no_clean_face - no_adv_face
            
            for i in range(to_remove):
                adv_train_set.pop()
        
        elif(no_adv_face > no_clean_face):
            to_remove = no_adv_face - no_clean_face
            
            for i in range(to_remove):
                clean_train_set.pop()
        
        clean_train_set = torch.stack(clean_train_set)
        torch.save(clean_train_set, 'dataset/clean_train_set.pt')
            
        adv_train_set = torch.stack(adv_train_set)
        torch.save(adv_train_set, 'dataset/adv_train_set.pt')

        dataloader_detector = DataLoader(TensorDataset(clean_train_set, adv_train_set), collate_fn=collate_fn)

        torch.save(dataloader_detector, 'dataset/dataloader_detector.pt')
        


We finally initialise and configure the `InceptionResnetV1` model, which serves as the backbone of the classification pipeline (`NN1`).

In [ ]:
from facenet_pytorch import InceptionResnetV1

# ——————————————————————————————————————————————————————
#   Initialize the pre-trained face-recognition model
# ——————————————————————————————————————————————————————

# We use the InceptionResnetV1 architecture from the facenet-pytorch package,
# pre-trained on the VGGFace2 dataset for high-quality face embeddings.
# By calling .eval(), we set the model to inference mode (disables dropout, batchnorm updates).
# We then move the model to the appropriate device (GPU if available, else CPU).
nn1 = InceptionResnetV1(
    pretrained='vggface2'  # load weights trained on the VGGFace2 face dataset
).eval().to(device)         # switch to evaluation mode and transfer to GPU/CPU

# —————————————————————————————————————————————
#           Enable classification head
# —————————————————————————————————————————————

# By default, InceptionResnetV1 returns 512-dimensional embeddings.
# Setting .classify instructs the model to append a linear classification
# layer on top of the embeddings, so that nn1(input) returns raw class logits
# for all identities in VGGFace2 (8 631 classes), instead of embeddings.
nn1.classify = True

Now we proceed to the configuration and purpose of the `classifier_nn1` object already used in the previous notebooks for the attack generation with compatibility with `ART` library.

In [ ]:
import torch.nn as nn
import torch.optim as optim

classifier_nn1 = PyTorchClassifier(
    model=nn1,                                                     # The PyTorch model to use
    clip_values=(-1, 1),                                           # The minimum and maximum values of the input
    loss=nn.CrossEntropyLoss(),                                    # The loss function
    optimizer=optim.Adam(nn1.parameters(), lr=0.01),               # The optimizer
    input_shape=(3, 160, 160),                                     # The shape of the input
    nb_classes=LABELS.size,                                        # The number of classes
    device_type='cuda' if torch.cuda.is_available() else 'cpu'
)

The next step involves setting up a `DataLoader` for the test set, which is essential for evaluating both the classifier and the detector. We use the `torchvision.datasets.ImageFolder` to load the test set images, applying standard resizing and tensor conversion transforms. This ensures consistency in input shapes for downstream evaluation.

If `MTCNN`-based alignment was previously applied (`mtcnn_processing` is `True`), the transform is set to `None`, as the images have already been preprocessed and saved in the expected format. We also create a reverse mapping `idx_to_class` to easily retrieve the class name for each sample during evaluation and visualization.

The `DataLoader` batches the images and provides efficient access to the test set, which is crucial for performance when evaluating adversarial robustness or conducting targeted attacks in later steps.

In [ ]:
from utils import *

# Define the transforms for the DataLoader
transforms = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
])

# Create a DataLoader for the test set by using the ImageFolder dataset
dataset = datasets.ImageFolder(root=test_set_data_folder, transform=transforms if not mtcnn_processing else None)
dataset.idx_to_class = {i: c.replace('', '') for i, c in enumerate(test_set['Name'])}
dataloader = DataLoader(dataset, collate_fn=collate_fn, num_workers=0)                                                                                                                           

Now we **load the aligned test set** previously created using MTCNN-based face detection and alignment. The dataset is stored as a PyTorch `DataLoader` object (`dataloader_aligned_nn1.pt`) and is restored directly to memory for subsequent use.

Next, we extract the **images (`x_test_aligned_nn1`)** and their corresponding **labels (`y_test_aligned_nn1`)** from the `DataLoader` into separate lists. These are then converted into stacked tensors, ensuring proper dimensionality for batch processing in downstream tasks.

Finally, the tensors are moved to the CPU and converted into NumPy arrays, making them compatible with other tools (e.g., ART's attacks and evaluations) and ready for adversarial attack generation or detector testing.

In [ ]:
# Load the aligned DataLoader
dataloader_aligned_nn1 = torch.load('dataset/dataloader_aligned_nn1.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
x_test_aligned_nn1, y_test_aligned_nn1 = zip(*[(sample[0], sample[1]) for sample in dataloader_aligned_nn1])

# Create the batches of the tensors of the images and labels
x_test_aligned_nn1 = torch.stack(x_test_aligned_nn1)
y_test_aligned_nn1 = torch.tensor(y_test_aligned_nn1)

x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy() 

We also load the `dataloader_detector.pt`, which contains the **aligned training data** for the binary adversarial detector. This dataset includes:

- **Clean samples (`clean_train_set`)**: genuine images from the training set.
- **Adversarial samples (`adv_train_set`)**: previously generated adversarial examples.

These sets are extracted and stacked into batch tensors, then converted to NumPy arrays to be compatible with downstream analysis and training steps.

In [ ]:
# Load the aligned DataLoader
dataloader_detector = torch.load('dataset/dataloader_detector.pt', weights_only=False)

# Get the x_test_aligned_nn1 and y_test_aligned_nn1 from the DataLoader
clean_train_set, adv_train_set = zip(*[(sample[0], sample[1]) for sample in dataloader_detector])

# Create the batches of the tensors of the images and labels
clean_train_set = torch.stack(clean_train_set)
adv_train_set = torch.stack(adv_train_set)

clean_train_set = clean_train_set.cpu().numpy() 
adv_train_set = adv_train_set.cpu().numpy()

class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

## 2. Detector of Adversarial Samples Defense Strategy

### Detector Training

Now, let us move on to the phase of creating the actual training set for the detector.

#### Training Set Configuration

In this step, we focus on constructing the dataset that will be used to train the detector for adversarial sample detection. To ensure consistency and high-quality data, we rely on the already aligned and pre-processed images: the clean samples and the adversarial samples.

This phase involves the follow main steps:

- **Loading the aligned images**: The clean and adversarial images are loaded from the previously saved PyTorch tensors (`clean_train_set.pt` and `adv_train_set.pt`). This guarantees that the detector works with the same aligned and cropped images as the original face-recognition pipeline.

- **Combining clean and adversarial samples**: Both sets are concatenated into a single training array, `x_train`, while assigning binary labels (`0` for clean samples and `1` for adversarial samples). The balanced dataset is essential for avoiding biases towards one class and ensures fair training.

- **Shuffling the dataset**: To avoid order-based biases during training, the dataset is shuffled randomly. This randomisation enhances the generalisation ability of the detector, exposing it to diverse samples in a random order.

- **Converting to the detector’s expected input size**: The images, originally prepared for NN1 at 160×160 resolution, are resized to 224×224 to meet the input size requirements of the ResNet50-based detector. This resizing step ensures seamless compatibility between the original feature extractor and the new detector model.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from art.defences.detector.evasion import BinaryInputDetector
from torchvision import models
from torchsummary import summary

detector_model = models.resnet50(weights='DEFAULT')

if torch.cuda.is_available():
    detector_model.cuda()

# Add a Linear layer to the classifier
detector_model.fc = nn.Linear(2048, 2, device=device)

for name, param in detector_model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

summary(detector_model, input_size=(3, 224, 224))

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# canale-wise clip bounds for the normalized input
min_val = (0.0 - mean) / std
max_val = (1.0 - mean) / std

detector_classifier = PyTorchClassifier(
    clip_values=(min_val, max_val), 
    model=detector_model, 
    loss=nn.CrossEntropyLoss(),                                    # The loss function
    optimizer=optim.Adam(filter(lambda p: p.requires_grad, detector_model.parameters()), lr=0.01),               # The optimizer
    input_shape=(3, 224, 224),                                     # The shape of the input
    nb_classes=2,                                        # The number of classes
    device_type='cuda' if torch.cuda.is_available() else 'cpu')

detector = BinaryInputDetector(detector_classifier)

#### Training Set Creation

Now let's proceed with the generation of the actual training set, specifically focusing on the adversarial samples.

This phase focuses on the creation of adversarial examples to support the robust training of an adversarial detector for the NN1 face recognition system. This part of the project, as said before, focuses solely on the generation of adversarial examples using the **Fast Gradient Sign Method (FGSM)**, the **Projected Gradient Descent (PGD)**, and the **Basic Iterative Method (BIM)**.

A fixed set of 1000 test identities (`test_names`) is loaded from a CSV file to support the generation of realistic targeted attacks, which emulate adversaries seeking to misclassify an image as a specific known identity. The dataset used for adversarial generation, `adv_train_set`, is carefully prepared beforehand to contain aligned and standardised image data compatible with the classifier.

1. For the `FGSM attack`, the dataset is divided to include both untargeted and targeted adversarial examples. This ensures that the detector is trained to recognise not only attacks that aim to degrade the classifier’s overall performance but also those specifically crafted to mislead it towards a particular identity. Multiple epsilon values are systematically explored to create adversarial examples of varying intensity, ranging from subtle distortions to more noticeable alterations. Random sampling within each configuration ensures that the generated examples remain diverse and unbiased, forming a balanced and realistic foundation for the detector’s training.

2. For the `PGD attack`, the same division between untargeted and targeted examples is implemented. In addition to this, the PGD attack is carried out with systematic variations in key parameters, such as epsilon, step size, number of iterations, and the use of multiple random initialisations. These variations simulate a range of realistic adversarial scenarios, from cautious, low-intensity attacks to more aggressive and persistent ones. This systematic approach, combined with random selection of images and target identities, guarantees that the adversarial dataset is comprehensive and representative of the challenges the detector might face in practice.

3. For the `BIM attack`, the division into untargeted and targeted samples is again maintained. The same parameter grid as PGD is explored, with adjustments in epsilon, step size, and number of iterations, but applied within the iterative refinement framework of BIM. This ensures consistency in dataset construction while capturing the specific iterative nature of BIM, which can introduce different perturbation characteristics compared to PGD. Random sampling across these configurations further enhances the diversity and representativeness of the adversarial examples, ensuring that the detector is exposed to a wide and realistic spectrum of adversarial inputs during its training process.

In [ ]:
import os
import numpy as np
import pandas as pd
from utils import *
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, BasicIterativeMethod
# —————————————————————————————————————————————
#    Config and pre-loaded objects
# —————————————————————————————————————————————
# classifier_nn1: PyTorchClassifier for face recognition model
# adv_train_set: NumPy array shape (N, C, H, W) of images to perturb
# y_true       : NumPy array shape (N,) with true label indices for adv_train_set
# LABELS       : np.ndarray of class names (strings)
# class_to_idx : dict mapping class name (string) -> index (int) for classifier
# test_set_annotations_folder: path to 'test_set.csv' with columns: Class_ID,Name,Sample_Num,Flag,Gender


# Create output directories
os.makedirs('dataset/detector_fgsm_samples', exist_ok=True)
os.makedirs('dataset/detector_bim_samples', exist_ok=True)
os.makedirs('dataset/detector_pgd_samples', exist_ok=True)

# Load test identities and class names (fixed list of 1000)
test_set = pd.read_csv(test_set_annotations_folder)
test_names = test_set['Name'].astype(str).tolist()

# Total samples in adv_train_set
N = 1500

# —————————————————————————————————————————————
#  1) FGSM on first 1500 samples
# —————————————————————————————————————————————
fgsm_total = 750
half = fgsm_total // 2  # 750 untargeted, 750 targeted
epsilons = [0.02, 0.04, 0.06, 0.08, 0.10]
per_eps = half // len(epsilons)  # 150 samples per ε per mode
idx_fgsm = np.arange(fgsm_total)
idx_unt = idx_fgsm[:half]
idx_tar = idx_fgsm[half:fgsm_total]

for mode, idx_mode in [('untargeted', idx_unt), ('targeted', idx_tar)]:
    for eps in epsilons:
        sel = np.random.choice(idx_mode, size=per_eps, replace=False)
        x_sub = adv_train_set[sel]

        if mode == 'untargeted':
            attack = FastGradientMethod(estimator=classifier_nn1, eps=eps, targeted=False)
            x_adv = attack.generate(x_sub)
        else:
            # Select a random target identity from the fixed 1000 names
            target_name = np.random.choice(test_names)
            # Generate one-hot labels using your provided function
            one_hot_targets = generate_one_hot_target_and_plot_image(
                target_name=target_name,
                LABELS=LABELS,
                class_to_idx=class_to_idx,
                x_samples=x_sub,
                all_images=x_test_aligned_nn1
            )
            attack = FastGradientMethod(estimator=classifier_nn1, eps=eps, targeted=True)
            x_adv = attack.generate(x_sub, y=one_hot_targets)

        fname = f'fgsm_{mode}_eps{eps:.2f}.npy'
        np.save(os.path.join('dataset/detector_fgsm_samples', fname), x_adv)
        print(f"[FGSM {mode}] eps={eps:.2f} -> {x_adv.shape[0]} samples saved to {fname}")


# —————————————————————————————————————————————
#  2) PGD on remaining samples
# —————————————————————————————————————————————

idx_pgd = np.arange(fgsm_total, N)
eps_list = [0.02, 0.06, 0.10]
modes = ['untargeted', 'targeted']
per_cell = (N - fgsm_total) // (len(eps_list) * 2 * 2 * len(modes))  # ~62
num_random_init = 5

for eps in eps_list:
    for frac in [2, 4]:
        alpha = eps / frac
        for steps in [10, 20]:
            for mode in modes:
                sel = np.random.choice(idx_pgd, size=per_cell, replace=False)
                x_sub = adv_train_set[sel]

                if mode == 'untargeted':
                    attack = ProjectedGradientDescent(
                        estimator=classifier_nn1,
                        eps=eps,
                        eps_step=alpha,
                        max_iter=steps,
                        num_random_init=num_random_init,
                        targeted=False
                    )
                    x_adv = attack.generate(x_sub)
                else:
                    # Select a random target identity from the fixed 1000 names
                    target_name = np.random.choice(test_names)
                    one_hot_targets = generate_one_hot_target_and_plot_image(
                        target_name=target_name,
                        LABELS=LABELS,
                        class_to_idx=class_to_idx,
                        x_samples=x_sub,
                        all_images=x_test_aligned_nn1
                    )
                    attack = ProjectedGradientDescent(
                        estimator=classifier_nn1,
                        eps=eps,
                        eps_step=alpha,
                        max_iter=steps,
                        num_random_init=num_random_init,
                        targeted=True
                    )
                    x_adv = attack.generate(x_sub, y=one_hot_targets)

                fname = f'pgd_eps{eps:.2f}_a{alpha:.3f}_T{steps}_{mode}.npy'
                np.save(os.path.join('dataset/detector_pgd_samples', fname), x_adv)
                print(f"[PGD {mode}] eps={eps:.2f}, alpha={alpha:.3f}, T={steps}, init={num_random_init} -> {x_adv.shape[0]} samples in {fname}")
                


# —————————————————————————————————————————————
#  2) BIM on remaining samples
# —————————————————————————————————————————————

number = len(adv_train_set) - N
idx_bim = np.arange(N, N + number)
eps_list = [0.02, 0.06, 0.10]
modes = ['untargeted', 'targeted']
per_cell = number // (len(eps_list) * 2 * 2 * len(modes))  

for eps in eps_list:
    for frac in [2, 4]:
        alpha = eps / frac
        for steps in [10, 20]:
            for mode in modes:
                sel = np.random.choice(idx_pgd, size=per_cell, replace=False)
                x_sub = adv_train_set[sel]

                if mode == 'untargeted':
                    attack = BasicIterativeMethod(
                        estimator=classifier_nn1,
                        eps=eps,
                        eps_step=alpha,
                        max_iter=steps,
                        targeted=False
                    )
                    x_adv = attack.generate(x_sub)
                else:
                    # Select a random target identity from the fixed 1000 names
                    target_name = np.random.choice(test_names)
                    one_hot_targets = generate_one_hot_target_and_plot_image(
                        target_name=target_name,
                        LABELS=LABELS,
                        class_to_idx=class_to_idx,
                        x_samples=x_sub,
                        all_images=x_test_aligned_nn1
                    )
                    attack = BasicIterativeMethod(
                        estimator=classifier_nn1,
                        eps=eps,
                        eps_step=alpha,
                        max_iter=steps,
                        targeted=True
                    )
                    x_adv = attack.generate(x_sub, y=one_hot_targets)

                fname = f'bim_eps{eps:.2f}_a{alpha:.3f}_T{steps}_{mode}.npy'
                np.save(os.path.join('dataset/detector_bim_samples', fname), x_adv)
                print(f"[BIM {mode}] eps={eps:.2f}, alpha={alpha:.3f}, T={steps} -> {x_adv.shape[0]} samples in {fname}")

To ensure that the final dataset used for training the adversarial detector is precisely balanced and meets the required size, a careful verification and adjustment process is implemented for both PGD and BIM adversarial examples. After the initial batch-based generation of adversarial samples, all existing `.npy` files within each corresponding attack’s output directory are loaded and concatenated into a single dataset. This guarantees that all previously generated samples are gathered together in a consistent manner.

The concatenated dataset is then compared against the target number of adversarial examples needed. If there is a shortfall, additional adversarial examples are generated to fill the gap. This is done by selecting images and attack parameters at random, respecting the same variability and balance as in the main attack generation loops. Random sampling is used to ensure that these additional samples are diverse and not biased towards any particular attack configuration.  

This methodical approach is repeated for both PGD and BIM attacks to guarantee that the final datasets strictly adhere to the target sizes (`desired` for PGD and `number` for BIM). This ensures that the dataset used for training the detector remains consistent and comprehensive, covering a broad spectrum of adversarial manipulations.

In [ ]:
import numpy as np
import glob

pgd_folder = 'dataset/detector_pgd_samples'
pgd_files  = sorted(glob.glob(f'{pgd_folder}/pgd_*.npy'))
pgd_batches = [np.load(f) for f in pgd_files]
all_pgd = np.concatenate(pgd_batches, axis=0)
current_count = all_pgd.shape[0]
desired = 750

if current_count < desired:
    missing = desired - current_count
    print(f"Generating {missing} extra PGD samples to reach {desired}")
    eps_list = [0.02, 0.06, 0.10]
    alpha_fracs = [2, 4]
    steps_list = [10, 20]
    modes = ['untargeted', 'targeted']
    for _ in range(missing):
        i = np.random.choice(idx_pgd)
        eps = np.random.choice(eps_list)
        alpha = eps / np.random.choice(alpha_fracs)
        steps = np.random.choice(steps_list)
        mode = np.random.choice(modes)

        x0 = adv_train_set[i:i+1]

        if mode == 'untargeted':
            atk = ProjectedGradientDescent(
                estimator=classifier_nn1,
                eps=eps,
                eps_step=alpha,
                max_iter=steps,
                num_random_init=1,
                targeted=False
            )
            x1 = atk.generate(x0)
        else:
            target_name = np.random.choice(test_names)
            y1 = generate_one_hot_target_and_plot_image(
                target_name=target_name,
                LABELS=LABELS,
                class_to_idx=class_to_idx,
                x_samples=x0,
                all_images=x_test_aligned_nn1
            )
            atk = ProjectedGradientDescent(
                estimator=classifier_nn1,
                eps=eps,
                eps_step=alpha,
                max_iter=steps,
                num_random_init=1,
                targeted=True
            )
            x1 = atk.generate(x0, y=y1)
        all_pgd = np.concatenate([all_pgd, x1], axis=0)

np.save(f'{pgd_folder}/all_pgd.npy', all_pgd)


In [ ]:
import numpy as np
import glob

bim_folder = 'dataset/detector_bim_samples'
bim_files  = sorted(glob.glob(f'{bim_folder}/bim_*.npy'))
bim_batches = [np.load(f) for f in bim_files]
all_bim = np.concatenate(bim_batches, axis=0)
current_count = all_bim.shape[0]
desired = number

if current_count < desired:
    missing = desired - current_count
    print(f"Generating {missing} extra BIM samples to reach {desired}")
    eps_list = [0.02, 0.06, 0.10]
    alpha_fracs = [2, 4]
    steps_list = [10, 20]
    modes = ['untargeted', 'targeted']
    for _ in range(missing):
        i = np.random.choice(idx_bim)
        eps = np.random.choice(eps_list)
        alpha = eps / np.random.choice(alpha_fracs)
        steps = np.random.choice(steps_list)
        mode = np.random.choice(modes)

        x0 = adv_train_set[i:i+1]

        if mode == 'untargeted':
            atk = BasicIterativeMethod(
                estimator=classifier_nn1,
                eps=eps,
                eps_step=alpha,
                max_iter=steps,
                targeted=False
            )
            x1 = atk.generate(x0)
        else:
            target_name = np.random.choice(test_names)
            y1 = generate_one_hot_target_and_plot_image(
                target_name=target_name,
                LABELS=LABELS,
                class_to_idx=class_to_idx,
                x_samples=x0,
                all_images=x_test_aligned_nn1
            )
            atk = BasicIterativeMethod(
                estimator=classifier_nn1,
                eps=eps,
                eps_step=alpha,
                max_iter=steps,
                targeted=True
            )
            x1 = atk.generate(x0, y=y1)
        all_bim = np.concatenate([all_bim, x1], axis=0)

np.save(f'{bim_folder}/all_bim.npy', all_bim)

Finally, all adversarial datasets (FGSM, PGD, and BIM) are loaded and concatenated to form a single, unified adversarial dataset. This dataset is then combined with the clean images to produce the final training dataset for the adversarial detector. Binary labels are assigned—`0` for clean images and `1` for adversarial examples—and the dataset is shuffled to avoid any ordering bias.

The transformed data, adjusted in batches to the input requirements of the detector, is ready for training.

In [ ]:
import glob

device = 'cuda' if torch.cuda.is_available() else 'cpu'
fgsm_files = sorted(glob.glob('dataset/detector_fgsm_samples/*.npy'))
pgd_folder = 'dataset/detector_pgd_samples'
bim_folder = 'dataset/detector_bim_samples'

all_fgsm = np.concatenate([np.load(f) for f in fgsm_files], axis=0)
all_bim  = np.load(f'{bim_folder}/all_bim.npy')
all_pgd  = np.load(f'{pgd_folder}/all_pgd.npy')
all_adv  = np.concatenate([all_fgsm, all_bim, all_pgd], axis=0)

clean = clean_train_set  # shape: (N_clean, C, H, W)
adv   = all_adv          # shape: (N_adv,   C, H, W)

x_train = np.concatenate([clean, adv], axis=0)  # shape: (N_clean + N_adv, C, H, W)

y_clean = np.zeros(len(clean), dtype=np.int64)
y_adv   = np.ones(len(adv),   dtype=np.int64)
y_train = np.concatenate([y_clean, y_adv], axis=0)  # shape: (N_clean + N_adv,)


idx = np.arange(len(y_train))
np.random.shuffle(idx)
x_train = x_train[idx]
y_train = y_train[idx]

conv_batch_size = 128

x_transformed = conversion_nn1_to_detector(x_train, batch_size=conv_batch_size)

#### Training and Saving

After assembling the final training dataset and transforming it into the appropriate input format for the adversarial detector, the detector model is trained using a standard training procedure with a batch size of 32 and for 30 epochs. This ensures that the model is exposed to a sufficient number of iterations over the data to learn to distinguish between clean and adversarial examples effectively.

In [ ]:
detector.fit(x_transformed, y_train, batch_size=32, nb_epochs=30)

Upon completion of training, the model’s weights are saved to a file, `detector_model.pth`, enabling future use without the need for retraining. Optionally, a set of relevant parameters—such as the clip values, input shape, and number of classes—are also stored in a separate file, `detector_params.pkl`, using the `pickle` module. This allows the detector to be restored or deployed quickly and consistently, ensuring reproducibility and simplifying integration into future testing or evaluation pipelines.

In [ ]:
torch.save(detector_model.state_dict(), 'detector_model.pth')

import pickle

params = {
    'clip_values': (min_val, max_val),
    'input_shape': (3, 224, 224),
    'nb_classes': 2,
}
with open('detector_params.pkl', 'wb') as f:
    pickle.dump(params, f)


### Detector Testing

To prepare for testing and evaluation of the adversarial detector, the previously trained detector model is reconstructed by first creating a fresh instance of the model architecture and then loading the saved weights. The pre-trained `resnet50` backbone is adjusted by replacing the final layer to match the binary classification task of detecting adversarial versus clean images. The model is set to evaluation mode to ensure consistent inference behaviour.

The configuration parameters used during the original training process, such as input scaling (`clip_values`), input shape, and number of classes, are loaded from the previously saved configuration file. This ensures that the reconstructed classifier (`detector_classifier`) mirrors the exact settings of the trained detector and maintains consistency in preprocessing and predictions.

The testing phase then focuses on evaluating the detector’s performance on a carefully selected subset of identities. This is achieved by selecting specific class names of interest and mapping them to their corresponding indices in the dataset. Any missing class names are identified and reported, maintaining clarity and consistency in the evaluation set-up.

This preparatory step ensures that the testing phase is conducted in an environment that replicates the original training conditions, allowing for reliable and consistent evaluation of the adversarial detector’s effectiveness.

In [ ]:
import pickle

detector_model = models.resnet50(weights='DEFAULT')
detector_model.fc = nn.Linear(2048, 2)
detector_model.load_state_dict(torch.load("detector_model.pth"))
detector_model.eval()

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

if torch.cuda.is_available():
    detector_model.cuda()

with open("detector_params.pkl", "rb") as f:
    config = pickle.load(f)

detector_classifier = PyTorchClassifier(
    clip_values=config["clip_values"],
    model=detector_model,
    loss=nn.CrossEntropyLoss(),
    optimizer=optim.Adam(filter(lambda p: p.requires_grad, detector_model.parameters()), lr=0.01),
    input_shape=config["input_shape"],
    nb_classes=config["nb_classes"],
    device_type='cuda' if torch.cuda.is_available() else 'cpu'
)

detector = BinaryInputDetector(detector_classifier)

selected_classes = ['Andrea_Bocelli','Gigi_DAlessio','Diego_Abatantuono','Diego_Maradona','Francesco_Totti','Dries_Mertens']

class_to_idx = {v: k for k, v in dataset.idx_to_class.items()}

idx_test_images = []
for name in selected_classes:
    if name in class_to_idx:
        idx_test_images.append(class_to_idx[name])
    else:
        print(f"Name not found: {name}")

N_SAMPLES = len(x_test_aligned_nn1)

The aligned and preprocessed test images are first transformed to match the detector’s expected input format using the same batch-based conversion as in training. The detector classifier is then used to predict the probability of each image being adversarial. The number of images identified as adversarial is counted and displayed, providing an immediate, clear indication of how the trained detector performs when confronted with clean, unperturbed test data.

In [ ]:
x_test = conversion_nn1_to_detector(x_test_aligned_nn1, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Clean test data")

In [ ]:
if isinstance(y_test_aligned_nn1, torch.Tensor):
    y_test_aligned_nn1 = y_test_aligned_nn1.cpu().numpy()
    
if isinstance(x_test_aligned_nn1, torch.Tensor):
    x_test_aligned_nn1 = x_test_aligned_nn1.cpu().numpy()

#### FGSM (Fast Gradient Sign Method) Adversarial Attack

Now, we proceed with the evaluation of the trained adversarial detector against adversarial examples generated using the `FGSM attack`. This testing phase allows to access the detector’s ability to distinguish clean samples from those that have been specifically perturbed to degrade classifier performance.

In [ ]:
from art.attacks.evasion import FastGradientMethod

fgsm_adv_folder = os.path.join(adversarial_folder, 'FGSM_nn1')
os.makedirs(fgsm_adv_folder, exist_ok=True)

y_true = torch.load('y_true.pt')

##### Error Generic

The `previously generated FGSM Error Generic adversarial samples` are loaded and converted to the detector’s expected input format. The detector classifier then processes these inputs, classifying each as either adversarial or clean.

In [ ]:
x_test_adv_fgsm_g = torch.load(fgsm_adv_folder + '/x_test_adv_fgsm_g.pt')

x_test = conversion_nn1_to_detector(x_test_adv_fgsm_g, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Adversarial test data", attack_name="FGSM - Error Generic")

Following this, a more comprehensive security evaluation is performed by systematically varying the epsilon parameter of the FGSM attack, as done in the previous notebook. This approach assesses the detector’s sensitivity to different levels of perturbation strength.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Generic)
# ──────────────────────────────────────────────────────────────────────────────
eps_range = [0.01, 0.02, 0.03, 0.05, 0.1]

attacker = FastGradientMethod(classifier_nn1, eps=0.05)

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps in eps_range:
    attacker.set_params(**{'eps': eps})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - FGSM (Error-Generic)'
)

##### Error Specific

Now the `FGSM Error Specific` adversarial samples are then loaded and converted to the input format expected by the adversarial detector. Once converted, as before, the detector processes these images to determine which are adversarial and which are clean.

In [ ]:
x_test_adv_fgsm_s = torch.load(fgsm_adv_folder + '/x_test_adv_fgsm_s.pt')

x_test = conversion_nn1_to_detector(x_test_adv_fgsm_s, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Adversarial test data", attack_name="FGSM - Error Specific")

Continuing the evaluation of the adversarial detector with the FGSM Error Specific attack, a set of targeted adversarial examples is generated by selecting “Dave Mustaine” as the forced target identity. The epsilon parameter is varied systematically across a defined range to observe how detection performance and misclassification rates evolve under increasingly stronger perturbations. This analysis complements the earlier security evaluation for Error Generic adversarial samples and deepens our understanding of the detector’s robustness against targeted adversarial manipulations.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Security Evaluation Curve: FGSM (Error Specific)
# ──────────────────────────────────────────────────────────────────────────────

one_hot_targeted_label = generate_one_hot_target_and_plot_image("Dave_Mustaine", LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

eps_range = [0.01, 0.02, 0.03, 0.05, 0.1]

attacker = FastGradientMethod(classifier_nn1, eps=0.05, targeted=True)

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps in eps_range:
    attacker.set_params(**{'eps': eps})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - FGSM (Error-Specific)'
)

#### BIM (Basic Iterative Method) Adversarial Attack 

We now continue the analysis with `BIM`, extending the evaluation to this iterative attack method.

In [ ]:
from art.attacks.evasion import BasicIterativeMethod

bim_adv_folder = os.path.join(adversarial_folder, 'BIM_nn1')
os.makedirs(bim_adv_folder, exist_ok=True)

y_true = torch.load('y_true.pt')

##### Error Generic

The previous `BIM Error Generic adversarial examples` are then loaded and prepared for evaluation. Once transformed to match the detector’s input format, these examples are classified by the detector to determine which images are adversarial and which remain clean.

In [ ]:
x_test_adv_bim_g = torch.load(bim_adv_folder + '/x_test_adv_bim_g.pt')

x_test = conversion_nn1_to_detector(x_test_adv_bim_g, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Adversarial test data", attack_name="BIM - Error Generic")

Continuing the thorough analysis of the adversarial detector’s performance, as done previously, a comprehensive set of security evaluation curves is generated to examine how detection accuracy and classifier misclassification rates vary under different BIM attack configurations.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - BIM (Error Generic)
# —————————————————————————————————————————————————————————————

eps_range        = [0.01, 0.03, 0.05, 0.1]
eps_step_range   = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_range   = [1, 3, 5, 10, 15]

epsilon          = 0.03
epsilon_step     = 0.01
max_iter         = 3

attacker = BasicIterativeMethod(classifier_nn1, eps=epsilon, eps_step=epsilon_step, max_iter=max_iter)

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps in eps_range:
    attacker.set_params(**{'eps': eps})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - BIM (Varying Epsilon)',
    xlabel='Epsilon',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red'
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps_step in eps_step_range:
    attacker.set_params(**{'eps': epsilon, 'eps_step': eps_step})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_step_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - BIM (Varying Epsilon Step)',
    xlabel='Epsilon Step',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red',
    x_points=5
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for iter in max_iter_range:
    attacker.set_params(**{'eps_step': epsilon_step, 'max_iter': iter})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)
    
    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=max_iter_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - BIM (Varying Max Iterations)',
    xlabel='Max Iterations',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red',
    x_points=6
)


##### Error Specific

We now extend the evaluation to the BIM Error Specific adversarial examples, focusing on how the detector responds to these targeted perturbations. This phase complements the previous tests on untargeted attacks and completes the comprehensive security assessment of the detector against the different BIM attack scenarios.

In [ ]:
x_test_adv_bim_s = torch.load(bim_adv_folder + '/x_test_adv_bim_s.pt')

x_test = conversion_nn1_to_detector(x_test_adv_bim_s, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Adversarial test data", attack_name="BIM - Error Specific")

And now we extend the comprehensive security evaluation to BIM targeted attacks, focusing on how detection performance varies as the key attack parameters — epsilon, epsilon step size, and number of iterations — are systematically adjusted.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - BIM (Error Specific - Targeted)
# —————————————————————————————————————————————————————————————

one_hot_targeted_label = generate_one_hot_target_and_plot_image("Dave_Mustaine", LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

eps_range       = [0.01, 0.03, 0.05, 0.1]
eps_step_range  = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_range  = [1, 3, 5, 10, 15]

epsilon         = 0.03
epsilon_step    = 0.01
max_iter        = 3

attacker = BasicIterativeMethod(classifier_nn1, eps=epsilon, eps_step=epsilon_step, max_iter=max_iter, targeted=True)

# —————————————————————————————————————————————————————————————
# 1) Accuracy & Targeted Accuracy vs ε
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps in eps_range:
    attacker.set_params(**{'eps': eps})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - BIM Targeted (Varying Epsilon)',
    xlabel='Epsilon',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange'
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy & Targeted Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps_step in eps_step_range:
    attacker.set_params(**{'eps': epsilon, 'eps_step': eps_step})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_step_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - BIM Targeted (Varying Epsilon Step)',
    xlabel='Epsilon Step',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange',
    x_points=5
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy & Targeted Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for iter in max_iter_range:
    attacker.set_params(**{'eps_step': epsilon_step, 'max_iter': iter})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=max_iter_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - BIM Targeted (Varying Max Iterations)',
    xlabel='Max Iterations',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange',
    x_points=6
)

#### PGD (Projected Gradient Descent) Adversarial Attack

We now introduce PGD as the final attack considered in this evaluation phase. Its inclusion completes the suite of adversarial attacks explored, providing a comprehensive assessment of the detector’s performance across both simple and complex perturbation strategies.

In [ ]:
from art.attacks.evasion import ProjectedGradientDescent


pgd_adv_folder = os.path.join(adversarial_folder, 'PGD_nn1')
os.makedirs(pgd_adv_folder, exist_ok=True)

y_true = torch.load('y_true.pt')

##### Error Generic

We begin by evaluating the detector’s performance against the `PGD Error Generic adversarial examples`. These represent typical untargeted attacks generated by PGD and serve as an initial benchmark for assessing how well the detector can identify adversarial manipulations produced by this more sophisticated iterative attack.

In [ ]:
x_test_adv_pgd_g = torch.load(pgd_adv_folder + '/x_test_adv_pgd_g.pt')

x_test = conversion_nn1_to_detector(x_test_adv_pgd_g, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Adversarial test data", attack_name="PGD - Error Generic")

The evaluation of the adversarial detector’s performance continues with an extensive set of security evaluation curves for PGD, the most complex of the attacks considered.

In [ ]:
# —————————————————————————————————————————————————————————————
# Security Evaluation Curves - PGD (Error Generic)
# —————————————————————————————————————————————————————————————

eps_range           = [0.01, 0.03, 0.05, 0.10]
eps_step_range      = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_range      = [1, 3, 5, 10, 15]
num_random_range    = [1, 3, 5]

epsilon             = 0.03
epsilon_step        = 0.01
max_iter            = 3
num_random_init     = 5

attacker = ProjectedGradientDescent(classifier_nn1, eps=0.05, eps_step=0.01, max_iter=3, num_random_init=num_random_init)

# —————————————————————————————————————————————————————————————
# 1) Accuracy vs ε
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps in eps_range:
    attacker.set_params(**{'eps': eps})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD (Varying Epsilon)',
    xlabel='Epsilon',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red'
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps_step in eps_step_range:
    attacker.set_params(**{'eps': epsilon, 'eps_step': eps_step})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)
    
    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_step_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD (Varying Epsilon Step)',
    xlabel='Epsilon Step',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red',
    x_points=5
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for iter in max_iter_range:
    attacker.set_params(**{'eps_step': epsilon_step, 'max_iter': iter})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=max_iter_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD (Varying Max Iterations)',
    xlabel='Max Iterations',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red',
    x_points=6
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for num_random in num_random_range:
    attacker.set_params(**{'max_iter': iter, 'num_random_init': num_random})
    x_test_adv = attacker.generate(x_test_aligned_nn1)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=num_random_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD (Varying Random Initializations)',
    xlabel='Number of Random Initializations',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='navy',
    color2='red',
    x_points=4
)

##### Error Specific

The evaluation concludes with the targeted `PGD Error Specific attack`. These experiments build on the previously explored Error Generic scenarios by introducing targeted misclassification as an explicit adversarial goal.


In [ ]:
x_test_adv_pgd_s = torch.load(pgd_adv_folder + '/x_test_adv_pgd_s.pt')

x_test = conversion_nn1_to_detector(x_test_adv_pgd_s, batch_size=128)

flag_adv = np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)

print_detector_results(flag_adv, N_SAMPLES, label="Adversarial test data", attack_name="PGD - Error Specific")

Finally a comprehensive set of security evaluation curves is generated by varying the key attack parameters, mirroring the analysis structure used for other attacks. This ensures that the detector’s performance is rigorously tested against the full spectrum of adversarial manipulations introduced by PGD.

In [ ]:
# —————————————————————————————————————————————————————————————
# PGD Targeted - Security Evaluation Curves (Error Specific)
# —————————————————————————————————————————————————————————————

one_hot_targeted_label = generate_one_hot_target_and_plot_image("Fernando_Torres", LABELS, class_to_idx, x_test_aligned_nn1, x_test_aligned_nn1)

eps_range           = [0.01, 0.03, 0.05, 0.10]
eps_step_range      = [0.001, 0.005, 0.01, 0.02, 0.03]
max_iter_range      = [1, 3, 5, 10, 15]
num_random_range    = [1, 3, 5]

epsilon = 0.03
epsilon_step = 0.01
max_iter = 3
num_random_init = 5

attacker = ProjectedGradientDescent(classifier_nn1, eps=epsilon, eps_step=epsilon_step, max_iter=max_iter, num_random_init=num_random_init, targeted=True)

nb_flag_adv = []
nb_missclass = []

# —————————————————————————————————————————————————————————————
# 1) Accuracy & Targeted Accuracy vs ε
# —————————————————————————————————————————————————————————————
for eps in eps_range:
    attacker.set_params(**{'eps': eps})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD Targeted (Varying Epsilon)',
    xlabel='Epsilon',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange'
)

# —————————————————————————————————————————————————————————————
# 2) Accuracy & Targeted Accuracy vs ε_step
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for eps_step in eps_step_range:
    attacker.set_params(**{'eps': epsilon, 'eps_step': eps_step})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=eps_step_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD Targeted (Varying Epsilon Step)',
    xlabel='Epsilon Step',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange'
)

# —————————————————————————————————————————————————————————————
# 3) Accuracy & Targeted Accuracy vs max_iter
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for iter in max_iter_range:
    attacker.set_params(**{'eps_step': epsilon_step, 'max_iter': iter})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=max_iter_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD Targeted (Varying Max Iterations)',
    xlabel='Max Iterations',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange'
)

# —————————————————————————————————————————————————————————————
# 4) Accuracy & Targeted Accuracy vs num_random_init
# —————————————————————————————————————————————————————————————
nb_flag_adv = []
nb_missclass = []

for num_random in num_random_range:
    attacker.set_params(**{'max_iter': max_iter, 'num_random_init': num_random})
    x_test_adv = attacker.generate(x_test_aligned_nn1, one_hot_targeted_label)
    
    x_test = conversion_nn1_to_detector(x_test_adv, batch_size=128)

    nb_flag_adv += [np.sum(np.argmax(detector_classifier.predict(x_test), axis=1) == 1)]
    nb_missclass += [np.sum(LABELS[np.argmax(classifier_nn1.predict(x_test_adv), axis=1)] != y_true)]

plot_detector_security_evaluation_curve(
    range=num_random_range,
    nb_flag_adv=nb_flag_adv,
    nb_missclass=nb_missclass,
    N_SAMPLES=N_SAMPLES,
    title='Security Evaluation Curve - PGD Targeted (Varying Random Initializations)',
    xlabel='Number of Random Initializations',
    legend1='Adversarial samples detected (%)',
    legend2='Classifier misclassification rate (%)',
    color1='purple',
    color2='orange'
)